# Backtest Framework

Train and evaluate multiple models across rolling fiscal months for each brand/model combination.

In [1]:
# Imports
import sys
import warnings
from pathlib import Path
from functools import partial
from datetime import datetime

import pandas as pd
import numpy as np
import joblib

from sklearn.model_selection import cross_val_score, RepeatedKFold
from sklearn.metrics import make_scorer, mean_squared_error

# Suppress known warnings from edge cases in lr_models and statsmodels
warnings.filterwarnings('ignore', category=FutureWarning)
warnings.filterwarnings('ignore', category=RuntimeWarning)
# warnings.filterwarnings('ignore', message='Estimated scale is 0.0')  # RLM perfect fit warning

# Add project root to path
sys.path.insert(0, str(Path.cwd().parent))

from functions import lr_models as lr

In [2]:
# Configuration
config = {
    'run': '202609_3P',           # unique name of the training run
    'backtest': '202609_3P'         # unique name of the backtest
}

TRAIN_DATA_PATH = Path(f"../data/train/train_{config['run']}.csv")
PREDICT_DATA_PATH = Path(f"../data/predict/predict_{config['run']}.csv")
FISCAL_CAL_PATH = Path('../data/meta/fiscal_calendar.csv')
OUTPUT_DIR = Path('../data/backtest')

# Current fiscal month to backtest up to
CURRENT_FISCAL_MONTH = '2026-06'  # Format: YYYY-MM (fiscal year-month) -- Fiscal 2026-01 is February 2026
LOOKBACK_MONTHS = 6  # Number of previous months to include

# Window Strategy for training data
# - 'expanding': Use all available data before prediction month (original approach)
# - 'rolling': Use only the last ROLLING_WINDOW_MONTHS before prediction month
WINDOW_STRATEGY = 'rolling'  # 'expanding' or 'rolling'
ROLLING_WINDOW_MONTHS = 12     # Only used when WINDOW_STRATEGY = 'rolling' (12 = full fiscal year cycle)

# Model configurations: list of (label, base_estimator, pipeline_fn, params_dict)
# - label: display name / estimator label (can include variant suffix like 'RLM_Huber')
# - base_estimator: must match PIPELINE_MAP key for training (e.g., 'RLM')
# - pipeline_fn: factory function that creates the pipeline
# - params_dict: optional kwargs passed to pipeline_fn
MODEL_CONFIGS = [
    ('OLS', 'OLS', lr.pipeline_ols_std, {}),
    ('WLS', 'WLS', lr.pipeline_wls, {}),
    ('PLS', 'PLS', lr.pipeline_pls, {}),
    ('RLM_Tukey', 'RLM', lr.pipeline_rlm, {'norm':'tukey'}),
    ('PCA', 'PCA', lr.pipeline_pca_ols, {}),
]

In [3]:
# Load data
df_train_raw = pd.read_csv(TRAIN_DATA_PATH)
df_predict_raw = pd.read_csv(PREDICT_DATA_PATH)
df_fiscal = pd.read_csv(FISCAL_CAL_PATH)

# Parse dates
df_train_raw['weekstart'] = pd.to_datetime(df_train_raw['weekstart'])
df_fiscal['weekstart'] = pd.to_datetime(df_fiscal['weekstart'], format='%m/%d/%y')

# Merge to add fiscal info
df_train = df_train_raw.merge(
    df_fiscal[['weekstart', 'fyear', 'fmonth', 'fweek']],
    on='weekstart',
    how='left'
)

# Create fiscal_month key (e.g., '2025-02')
df_train['fiscal_month'] = df_train['fyear'].astype(str) + '-' + df_train['fmonth'].astype(str).str.zfill(2)

# Filter to only brand/model combinations we are interested in (prediction set for forecast month)
active_bc = df_predict_raw[['brand','model']].drop_duplicates()
df_train = df_train.merge(active_bc, on=['brand', 'model'], how='inner')

print(f"Training data shape: {df_train.shape}")
print(f"Fiscal months available: {sorted(df_train['fiscal_month'].unique())}")
print(f"Active brand/model combinations (in {CURRENT_FISCAL_MONTH}): {len(active_bc)}")

Training data shape: (2252, 53)
Fiscal months available: ['2024-01', '2024-02', '2024-03', '2024-04', '2024-05', '2024-06', '2024-07', '2024-08', '2024-09', '2024-10', '2024-11', '2024-12', '2025-01', '2025-02', '2025-03', '2025-04', '2025-05', '2025-06', '2025-07', '2025-08', '2025-09', '2025-10', '2025-11', '2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06', '2026-07']
Active brand/model combinations (in 2026-06): 35


In [4]:
# # TEMPORARY: Filter to subset of models for faster testing
# # Comment out or delete this cell for full run
TEST_MODELS = [
    ('brand us', 'Social | Conversion | CVR ASC Value'),
    ('brand us', 'Social | Conversion | CVR ASC Volume'),
    ('brand us', 'Social | Conversion | CVR ASC Omni'),
    ('brand outlet', 'Social | Conversion | CVR ASC Value'),
    ('brand outlet', 'Social | Conversion | CVR ASC Volume'),
    ('brand outlet', 'Social | Conversion | CVR ASC Omni'),
    ('brand ca', 'Social | Conversion | CVR ASC Value'),
    ('brand ca', 'Social | Conversion | CVR ASC Volume'),
    ('brand us', 'Search | Consideration | Non Brand'),
    ('brand us', 'Search | Conversion | Brand'),
    ('brand us', 'Search | Conversion | Non Brand'),
    ('brand us', 'Search | Conversion | PMAX'),
    ('brand us', 'Search | Conversion | RSC'),
    ('brand outlet', 'Search | Consideration | Non Brand'),
    ('brand outlet', 'Search | Conversion | Brand'),
    ('brand outlet', 'Search | Conversion | Non Brand'),
    ('brand outlet', 'Search | Conversion | PMAX'),
    ('brand outlet', 'Search | Conversion | RSC'),
    ('brand ca', 'Search | Consideration | Non Brand'),
    ('brand ca', 'Search | Conversion | Brand'),
    ('brand ca', 'Search | Conversion | Non Brand'),
    ('brand ca', 'Search | Conversion | PMAX'),
    ('brand ca', 'Search | Conversion | RSC')
]

df_train = df_train[df_train.apply(lambda r: (r['brand'], r['model']) in TEST_MODELS, axis=1)]
print(f"TESTING MODE: Filtered to {df_train[['brand', 'model']].drop_duplicates().shape[0]} models")

TESTING MODE: Filtered to 19 models


In [5]:
# Helper functions

def get_rolling_months(current_month: str, lookback: int = 5) -> list[str]:
    """
    Get list of fiscal months from current_month - lookback to current_month.
    
    Parameters
    ----------
    current_month : str
        Current fiscal month in 'YYYY-MM' format
    lookback : int
        Number of previous months to include
    
    Returns
    -------
    list[str]
        List of fiscal months in ascending order
    """
    all_months = sorted(df_train['fiscal_month'].unique())
    try:
        idx = all_months.index(current_month)
    except ValueError:
        raise ValueError(f"Month {current_month} not found in data. Available: {all_months}")
    
    start_idx = max(0, idx - lookback)
    return all_months[start_idx:idx + 1]


def filter_train_data(df: pd.DataFrame, cutoff_month: str, 
                      strategy: str = 'expanding', window_months: int = 13) -> pd.DataFrame:
    """
    Filter data to training window ending strictly before cutoff_month.
    
    Parameters
    ----------
    df : pd.DataFrame
        Data with fiscal_month column
    cutoff_month : str
        Fiscal month to predict (excluded from training)
    strategy : str
        'expanding' = all data before cutoff (growing window)
        'rolling' = only last window_months before cutoff (fixed window)
    window_months : int
        Number of months in rolling window (only used if strategy='rolling')
    
    Returns
    -------
    pd.DataFrame
        Filtered training data
    """
    if strategy == 'expanding':
        return df[df['fiscal_month'] < cutoff_month].copy()
    elif strategy == 'rolling':
        all_months = sorted(df['fiscal_month'].unique())
        try:
            cutoff_idx = all_months.index(cutoff_month)
        except ValueError:
            # cutoff_month not in data, find where it would be
            cutoff_idx = len([m for m in all_months if m < cutoff_month])
        start_idx = max(0, cutoff_idx - window_months)
        valid_months = all_months[start_idx:cutoff_idx]
        return df[df['fiscal_month'].isin(valid_months)].copy()
    else:
        raise ValueError(f"Unknown strategy: {strategy}. Use 'expanding' or 'rolling'.")


def filter_test_data(df: pd.DataFrame, target_month: str) -> pd.DataFrame:
    """Filter data to a single target month."""
    return df[df['fiscal_month'] == target_month].copy()


def prepare_Xy(df: pd.DataFrame, y_col: str = 'nd') -> tuple[pd.DataFrame, pd.Series]:
    """
    Prepare X and y for modeling by dropping non-feature columns.
    
    Returns
    -------
    tuple
        (X, y) where X is features and y is target
    """
    drop_cols = ['brand', 'model', 'weekstart', 'nd', 'FYear', 'FMonth', 'FWeek', 'fiscal_month']
    drop_cols = [c for c in drop_cols if c in df.columns]
    
    X = df.drop(columns=drop_cols)
    y = df[y_col]
    return X, y


# Test helper functions
test_months = get_rolling_months(CURRENT_FISCAL_MONTH, LOOKBACK_MONTHS)
print(f"Backtest months: {test_months}")
print(f"Window strategy: {WINDOW_STRATEGY}" + 
      (f" ({ROLLING_WINDOW_MONTHS} months)" if WINDOW_STRATEGY == 'rolling' else " (all available data)"))

Backtest months: ['2025-12', '2026-01', '2026-02', '2026-03', '2026-04', '2026-05', '2026-06']
Window strategy: rolling (12 months)


In [6]:
# Scoring functions

def mape(y_true, y_pred):
    """Mean Absolute Percentage Error."""
    y_true, y_pred = np.array(y_true), np.array(y_pred)
    # Avoid division by zero
    mask = y_true != 0
    if mask.sum() == 0:
        return np.nan
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100


def rmse(y_true, y_pred):
    """Root Mean Squared Error."""
    return np.sqrt(mean_squared_error(y_true, y_pred))


# Custom sklearn scorers (negative because sklearn maximizes)
mape_scorer = make_scorer(mape, greater_is_better=False)
rmse_scorer = make_scorer(rmse, greater_is_better=False)


def compute_scores(pipe, X: pd.DataFrame, y: pd.Series) -> dict:
    """
    Compute in-sample and cross-validated scores for a fitted pipeline.
    
    Returns
    -------
    dict
        Dictionary with RMSE, MAPE, R2, BP_stat, BP_p, VIF (in-sample and CV where applicable)
    """
    model = pipe.named_steps['model']
    results = model.results_
    
    # Get residuals
    if hasattr(model, 'resid_') and model.resid_ is not None:
        resid = model.resid_
    else:
        resid = results.resid if hasattr(results, 'resid') else None
    
    # In-sample predictions
    y_pred = pipe.predict(X)
    
    # In-sample scores
    rmse_is = rmse(y, y_pred)
    mape_is = mape(y, y_pred)
    
    # R-squared (handle models without rsquared_adj)
    if hasattr(results, 'rsquared_adj'):
        r2_is = results.rsquared_adj
    else:
        # Compute pseudo-R²
        ss_res = np.sum(resid ** 2) if resid is not None else np.nan
        ss_tot = np.sum((y - y.mean()) ** 2)
        r2_is = 1 - (ss_res / ss_tot) if ss_tot > 0 else 0
    
    # Breusch-Pagan (heteroskedasticity) - returns dict with 'statistic' and 'pvalue'
    bp = lr.calc_breusch_pagan(results, resid=resid)
    bp_stat = bp.get('statistic', np.nan)
    bp_p = bp.get('pvalue', np.nan)
    
    # VIF (multicollinearity)
    vif = lr.calc_vif(X)
    
    # Cross-validated scores
    CV_SPLITS = int(np.sqrt(results.nobs)) # adaptive k, sqrt approach
    cv = RepeatedKFold(n_splits=CV_SPLITS, n_repeats=10) # orig
    
    try:
        rmse_cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring=rmse_scorer)
        rmse_cv = -rmse_cv_scores.mean()  # Negate back
    except Exception:
        rmse_cv = np.nan
    
    try:
        mape_cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring=mape_scorer)
        mape_cv = -mape_cv_scores.mean()  # Negate back
    except Exception:
        mape_cv = np.nan
    
    try:
        r2_cv_scores = cross_val_score(pipe, X, y, cv=cv, scoring='r2')
        r2_cv = r2_cv_scores.mean()
    except Exception:
        r2_cv = np.nan
    
    return {
        'RMSE': round(rmse_is, 2),
        'RMSE_CV': round(rmse_cv, 2) if not np.isnan(rmse_cv) else np.nan,
        'MAPE': round(mape_is, 2),
        'MAPE_CV': round(mape_cv, 2) if not np.isnan(mape_cv) else np.nan,
        'R2': round(r2_is, 4),
        'R2_CV': round(r2_cv, 4) if not np.isnan(r2_cv) else np.nan,
        'BP_stat': round(bp_stat, 4) if not np.isnan(bp_stat) else np.nan,
        'BP_p': round(bp_p, 4) if not np.isnan(bp_p) else np.nan,
        'VIF': vif,
    }


print("Scoring functions defined.")

Scoring functions defined.


In [7]:
# Core backtest loop
import json

def run_backtest(df: pd.DataFrame, current_month: str, lookback: int = 5,
                 window_strategy: str = 'expanding', rolling_window: int = 13) -> tuple[list, list]:
    """
    Run backtest across rolling fiscal months for all brand/model combinations.
    
    Parameters
    ----------
    df : pd.DataFrame
        Full training data with fiscal_month column
    current_month : str
        Current fiscal month (endpoint of backtest)
    lookback : int
        Number of previous months to include in backtest evaluation
    window_strategy : str
        'expanding' = train on all data before prediction month
        'rolling' = train only on last rolling_window months
    rolling_window : int
        Number of months in rolling window (only used if strategy='rolling')
    
    Returns
    -------
    tuple[list, list]
        (backtest_scores, backtest_predictions) - lists of dictionaries
    """
    print(f"Window strategy: {window_strategy}" + 
          (f" ({rolling_window} months)" if window_strategy == 'rolling' else " (all available data)"))
    
    rolling_months = get_rolling_months(current_month, lookback)
    brand_models = df[['brand', 'model']].drop_duplicates().values.tolist()
    
    backtest_scores = []
    backtest_predictions = []
    
    total_iterations = len(rolling_months) * len(brand_models) * len(MODEL_CONFIGS)
    iteration = 0
    
    for fiscal_month in rolling_months:
        for brand, model in brand_models:
            # Filter to this brand/model
            df_bc = df[(df['brand'] == brand) & (df['model'] == model)]
            
            # Get train data (based on window strategy) and test (target month only)
            df_train_subset = filter_train_data(df_bc, fiscal_month, 
                                                strategy=window_strategy, 
                                                window_months=rolling_window)
            df_test_subset = filter_test_data(df_bc, fiscal_month)
            
            # Skip if insufficient data
            if len(df_train_subset) < 10 or len(df_test_subset) == 0:
                iteration += len(MODEL_CONFIGS)
                continue
            
            X_train, y_train = prepare_Xy(df_train_subset)
            X_test, y_test = prepare_Xy(df_test_subset)
            
            for model_label, base_estimator, pipeline_fn, model_params in MODEL_CONFIGS:
                iteration += 1
                print(f"Processing: {brand} / {model} - {model_label} ({fiscal_month})")
                
                # Convert params to JSON string for storage (empty dict -> empty string)
                params_str = json.dumps(model_params) if model_params else ''
                
                try:
                    # Create and fit pipeline with optional params
                    if model_params:
                        pipe = pipeline_fn(**model_params)
                    else:
                        pipe = pipeline_fn()
                    pipe.fit(X_train, y_train)
                    
                    # Compute scores
                    scores = compute_scores(pipe, X_train, y_train)
                    
                    # Store scores with metadata
                    # estimator: unique user-defined label (e.g., 'RLM_Tukey')
                    # base_estimator: estimator type for PIPELINE_MAP lookup (e.g., 'RLM')
                    backtest_scores.append({
                        'brand': brand,
                        'model': model,
                        'estimator': model_label,
                        'base_estimator': base_estimator,
                        'params': params_str,
                        'fiscal_month': fiscal_month,
                        **scores,
                        'pipeline': pipe,  # Store fitted pipeline
                    })
                    
                    # Generate predictions with confidence intervals
                    # Convert to numpy array for consistent positional indexing
                    y_pred = np.asarray(pipe.predict(X_test))
                    
                    # Get prediction intervals
                    # For pipelines with preprocessing, we need to transform X_test first
                    estimator_step = pipe.named_steps['model']
                    try:
                        # Check if pipeline has preprocessing step(s) before model
                        if 'col_trans' in pipe.named_steps:
                            X_test_transformed = pipe.named_steps['col_trans'].transform(X_test)
                        else:
                            X_test_transformed = X_test
                        
                        intervals = estimator_step.get_intervals(X_test_transformed, alpha=0.1)
                        ci_lo = intervals['obs_ci_lower'].values
                        ci_hi = intervals['obs_ci_upper'].values
                    except Exception:
                        # Some models may not support intervals
                        ci_lo = np.full(len(y_pred), np.nan)
                        ci_hi = np.full(len(y_pred), np.nan)
                    
                    # Store predictions by week
                    for i, (idx, row) in enumerate(df_test_subset.iterrows()):
                        backtest_predictions.append({
                            'brand': brand,
                            'model': model,
                            'estimator': model_label,
                            'base_estimator': base_estimator,
                            'params': params_str,
                            'fiscal_month': fiscal_month,
                            'fiscal_week': row['weekstart'],
                            'spend': row['spend'],
                            'revenue_actual': row['nd'],
                            'revenue_pred': y_pred[i],
                            'revenue_pred_ci_lo': ci_lo[i],
                            'revenue_pred_ci_hi': ci_hi[i],
                        })
                    
                except Exception as e:
                    print(f"Error: {brand}/{model}/{model_label}/{fiscal_month}: {e}")
                    continue
                
                # Progress update
                if iteration % 50 == 0:
                    print(f"Progress: {iteration}/{total_iterations} ({100*iteration/total_iterations:.1f}%)")
    
    print(f"Backtest complete: {len(backtest_scores)} model-months, {len(backtest_predictions)} predictions")
    return backtest_scores, backtest_predictions

In [8]:
# Run backtest
scores_list, predictions_list = run_backtest(
    df_train, CURRENT_FISCAL_MONTH, LOOKBACK_MONTHS,
    window_strategy=WINDOW_STRATEGY, 
    rolling_window=ROLLING_WINDOW_MONTHS
)

Window strategy: rolling (12 months)
Processing: brand us / Search | Conversion | Brand - OLS (2025-12)
Processing: brand us / Search | Conversion | Brand - WLS (2025-12)
Processing: brand us / Search | Conversion | Brand - PLS (2025-12)
Processing: brand us / Search | Conversion | Brand - RLM_Tukey (2025-12)
Processing: brand us / Search | Conversion | Brand - PCA (2025-12)
Processing: brand us / Search | Conversion | Non Brand - OLS (2025-12)
Processing: brand us / Search | Conversion | Non Brand - WLS (2025-12)
Processing: brand us / Search | Conversion | Non Brand - PLS (2025-12)
Processing: brand us / Search | Conversion | Non Brand - RLM_Tukey (2025-12)
Processing: brand us / Search | Conversion | Non Brand - PCA (2025-12)
Processing: brand us / Search | Conversion | PMAX - OLS (2025-12)
Processing: brand us / Search | Conversion | PMAX - WLS (2025-12)
Processing: brand us / Search | Conversion | PMAX - PLS (2025-12)
Processing: brand us / Search | Conversion | PMAX - RLM_Tukey (

In [9]:
# Create output DataFrames
df_scores = pd.DataFrame(scores_list)
df_predictions = pd.DataFrame(predictions_list)

# Display summary
print("Backtest Scores shape:", df_scores.shape)
print("\nScores columns:", df_scores.columns.tolist())
print("\nSample scores:")
df_scores.drop(columns=['pipeline']).head(10)

Backtest Scores shape: (665, 16)

Scores columns: ['brand', 'model', 'estimator', 'base_estimator', 'params', 'fiscal_month', 'RMSE', 'RMSE_CV', 'MAPE', 'MAPE_CV', 'R2', 'R2_CV', 'BP_stat', 'BP_p', 'VIF', 'pipeline']

Sample scores:


,brand,model,estimator,base_estimator,params,fiscal_month,RMSE,RMSE_CV,MAPE,MAPE_CV,R2,R2_CV,BP_stat,BP_p,VIF
0,brand us,Search | Conversion | Brand,OLS,OLS,,2025-12,62101.68,180095.34,4.82,10.16,0.9767,0.5875,21.4145,0.9997,inf
1,brand us,Search | Conversion | Brand,WLS,WLS,,2025-12,63851.17,161170.48,4.56,9.93,0.9128,0.4974,21.9365,0.9996,inf
2,brand us,Search | Conversion | Brand,PLS,PLS,,2025-12,83921.66,160159.40,6.08,8.49,0.9748,0.7488,5.2388,0.0728,inf
3,brand us,Search | Conversion | Brand,RLM_Tukey,RLM,"{""norm"": ""tukey""}",2025-12,63124.52,174671.38,4.61,10.13,0.9863,0.2420,23.1206,0.9991,inf
4,brand us,Search | Conversion | Brand,PCA,PCA,,2025-12,77685.73,161062.17,5.78,9.01,0.9698,0.3253,18.3157,0.3057,inf
5,brand us,Search | Conversion | Non Brand,OLS,OLS,,2025-12,1335.96,4567.70,14.74,57.45,0.8890,-0.8127,16.2592,1.0000,inf
6,brand us,Search | Conversion | Non Brand,WLS,WLS,,2025-12,1524.93,4035.51,11.62,53.19,0.8710,-1.4075,21.4478,0.9997,inf
7,brand us,Search | Conversion | Non Brand,PLS,PLS,,2025-12,1829.96,3805.03,24.06,44.74,0.9185,0.2380,0.9916,0.6091,inf
8,brand us,Search | Conversion | Non Brand,RLM_Tukey,RLM,"{""norm"": ""tukey""}",2025-12,1342.37,5381.76,14.11,57.45,0.9597,-1.7332,13.8895,1.0000,inf
9,brand us,Search | Conversion | Non Brand,PCA,PCA,,2025-12,1742.72,3747.99,20.56,43.07,0.8786,0.2479,7.8036,0.7308,inf


In [10]:
# Display predictions sample
print("Backtest Predictions shape:", df_predictions.shape)
print("\nPredictions columns:", df_predictions.columns.tolist())
print("\nSample predictions:")
df_predictions.head(10)

Backtest Predictions shape: (2850, 12)

Predictions columns: ['brand', 'model', 'estimator', 'base_estimator', 'params', 'fiscal_month', 'fiscal_week', 'spend', 'revenue_actual', 'revenue_pred', 'revenue_pred_ci_lo', 'revenue_pred_ci_hi']

Sample predictions:


,brand,model,estimator,base_estimator,params,fiscal_month,fiscal_week,spend,revenue_actual,revenue_pred,revenue_pred_ci_lo,revenue_pred_ci_hi
0,brand us,Search | Conversion | Brand,OLS,OLS,,2025-12,2026-01-04,8296.26,684015.62,626711.895805,361338.919690,892084.871920
1,brand us,Search | Conversion | Brand,OLS,OLS,,2025-12,2026-01-11,8219.70,748401.22,689701.883351,424312.522866,955091.243836
2,brand us,Search | Conversion | Brand,OLS,OLS,,2025-12,2026-01-18,9100.66,747553.87,724561.620994,480986.685111,968136.556878
3,brand us,Search | Conversion | Brand,OLS,OLS,,2025-12,2026-01-25,9481.32,710296.38,636199.271971,460836.961583,811561.582358
4,brand us,Search | Conversion | Brand,WLS,WLS,,2025-12,2026-01-04,8296.26,684015.62,651007.380407,470610.179066,831404.581748
5,brand us,Search | Conversion | Brand,WLS,WLS,,2025-12,2026-01-11,8219.70,748401.22,704861.409420,532890.358777,876832.460063
6,brand us,Search | Conversion | Brand,WLS,WLS,,2025-12,2026-01-18,9100.66,747553.87,736273.708291,577156.913968,895390.502614
7,brand us,Search | Conversion | Brand,WLS,WLS,,2025-12,2026-01-25,9481.32,710296.38,656956.075946,571844.154398,742067.997495
8,brand us,Search | Conversion | Brand,PLS,PLS,,2025-12,2026-01-04,8296.26,684015.62,657246.532765,508509.167921,805983.897609
9,brand us,Search | Conversion | Brand,PLS,PLS,,2025-12,2026-01-11,8219.70,748401.22,709431.270915,559514.114708,859348.427122


In [11]:
# Save outputs to timestamped folder
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
output_folder = OUTPUT_DIR / f"backtest_{config['backtest']}_{timestamp}"
output_folder.mkdir(parents=True, exist_ok=True)

# Save scores (with pipelines) as joblib
scores_path = output_folder / 'backtest_scores.joblib'
joblib.dump(df_scores, scores_path)
print(f"Saved scores to: {scores_path}")

# Save predictions as CSV
predictions_path = output_folder / 'backtest_predictions.csv'
df_predictions.to_csv(predictions_path, index=False)
print(f"Saved predictions to: {predictions_path}")

# Also save a scores CSV (without pipeline objects) for easy viewing
scores_csv_path = output_folder / 'backtest_scores.csv'
df_scores.drop(columns=['pipeline']).to_csv(scores_csv_path, index=False)
print(f"Saved scores CSV to: {scores_csv_path}")

Saved scores to: ../data/backtest/backtest_202609_3P_20260825_132820/backtest_scores.joblib
Saved predictions to: ../data/backtest/backtest_202609_3P_20260825_132820/backtest_predictions.csv
Saved scores CSV to: ../data/backtest/backtest_202609_3P_20260825_132820/backtest_scores.csv
